# Scraping Data Ulasan Duolingo Play Store (Indonesia)

Notebook ini mengambil data content aplikasi Duolingo secara mandiri dari internet menggunakan `google-play-scraper`.

Target:
- Minimal 10.000 sampel
- Data disimpan ke CSV untuk dipakai pada notebook analisis sentimen

Sumber data:
- Google Play Store

In [1]:
%pip install -q google-play-scraper pandas tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from google_play_scraper import Sort, reviews

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# Scrapping review aplikasi Duolingo
APP_ID = "com.duolingo"
APP_NAME = "DUOLINGO"
LANG = "id"
COUNTRY = "id"
TARGET_COUNT = 10000
MAX_BATCH = 2000

OUTPUT_CSV = "dataset_duolingo.csv"

print(f"Target total sampel: ~{TARGET_COUNT}".replace(",", "."))

Target total sampel: ~10000


c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def scrape_reviews_for_app(app_id: str, app_name: str, target_count: int = 2000):
    """Scrape review sampai target terpenuhi atau data habis."""
    all_rows = []
    token = None

    while len(all_rows) < target_count:
        batch_count = min(MAX_BATCH, target_count - len(all_rows))
        result, token = reviews(
            app_id,
            lang=LANG,
            country=COUNTRY,
            sort=Sort.NEWEST,
            count=batch_count,
            filter_score_with=None
        )

        if not result:
            break

        for r in result:
            all_rows.append(
                {
                    "user_name": r.get("userName"),
                    "score": r.get("score"),
                    "content": r.get("content"),
                }
            )

        if token is None:
            break

    return all_rows

In [4]:
all_reviews = scrape_reviews_for_app(APP_ID, APP_NAME, target_count=TARGET_COUNT)

raw_df = pd.DataFrame(all_reviews)
print(f"\nTotal content terkumpul (raw): {len(raw_df):,}".replace(",", "."))
raw_df.head()


Total content terkumpul (raw): 10.000


,user_name,score,content
0,Naufal Atha S,4,"Bagus sih tapi kadang pas udah nyelesain malah layarnya putih jadi harus keluar dulu biar bisa ngelanjutin, trus pas..."
1,Ni Made Shinta Hardiyanti bendesemas,5,Aplikasi yang paling bagus banget saya bisa belajar bahasa inggris yang benar.
2,Desmiati Buke,5,"duolinggo ini belajarnya seru apa lagi di catur bisa lawan oscar atau lawan orang lain.selain itu,kita bisa belajar ..."
3,Khoirul Codo,5,membantu saya belajar
4,BAYU SYAH PUTRA,1,Gk ada catur terus betanya penuh please klo mau bikin itu tuh yg bnr bukan saya marah tapi saya suka dengan game ini...


In [5]:
df = raw_df.copy()

print("Distribusi skor:")
display(df["score"].value_counts().sort_index())

# Simpan ke CSV
ordered_cols = [
    "user_name", "score", "content"
]
existing_cols = [c for c in ordered_cols if c in df.columns]
df[existing_cols].to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"\nFile CSV tersimpan di: {OUTPUT_CSV}")
print(f"Jumlah sampel akhir: {len(df):,}".replace(",", "."))

Distribusi skor:


score
1     485
2     180
3     350
4    1050
5    7935
Name: count, dtype: int64


File CSV tersimpan di: dataset_duolingo.csv
Jumlah sampel akhir: 10.000


In [6]:
df.sample(5, random_state=42)[["user_name", "score", "content"]]

,user_name,score,content
6252,shashie andriani,5,seru!!!!
4684,Shaqueena Nur Malaika,5,"cool, i love the characters ^_^ recommended for Everyone who cant do any languages :]"
1731,Dwi Fitri Anggraini,4,membantu belajar mandiri
4742,Tristan zaraski harun,5,aplikasi ini bagus buat belajar bahasa inggris
4521,Renan Lucu,5,gampang dan bisa nambah pendidikan
